# Lecture 4 · Notebook 2 — Positional encodings: sinusoids and rotations
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NLP-Course-CUHKSZ/NLP-course-cuhksz.github.io/blob/main/materials/tutorials/lecture04/lecture04_2_positional_encodings.ipynb)

**Companion to [Chapter 8 of the video lecture](https://nlp-course-cuhksz.github.io/lecture04.html#p8-s106)** (slides 106–111). Self-attention is blind to word order (Notebook 1),
so the Transformer must inject position. In this notebook you will:

1. build the binary-counter intuition of slide 107,
2. compute the original **sinusoidal** encoding and draw the picture of slide 108,
3. print the 8-dimensional table of slide 109 and add it to a word embedding,
4. check the property the authors point out: a shift by k is a **linear map**, and similarity depends only on the **distance**,
5. reproduce the **RoPE** example of slide 110 and verify that q·k depends only on the relative position.

CPU only, under a minute, no downloads.

In [ ]:
import math, numpy as np, torch
import matplotlib.pyplot as plt
np.set_printoptions(precision=2, suppress=True)

## 1 · A binary counter (slide 107)
Write positions 0–15 in binary. The lowest bit flips every step, the next every 2 steps, then every 4 and 8. Every position gets a
unique code built from "dimensions" that change at different rates.

In [ ]:
bits = np.array([[(p >> b) & 1 for b in (3, 2, 1, 0)] for p in range(16)])
for p, row in enumerate(bits):
    print(f"{p:2d}  " + " ".join(map(str, row)))

## 2 · The sinusoidal encoding (slide 106)
$$PE_{(pos,\,2i)} = \sin\!\big(pos / 10000^{2i/d}\big), \qquad PE_{(pos,\,2i+1)} = \cos\!\big(pos / 10000^{2i/d}\big)$$
Pair *i* oscillates with angular rate $\omega_i = 10000^{-2i/d}$: fast for the first pairs (the "low bits"), very slow for the last
ones (the "high bits"). It is a smooth, continuous version of the counter.

In [ ]:
def sinusoidal(n_pos, d):
    pos = np.arange(n_pos)[:, None]
    rate = 10000.0 ** (-np.arange(0, d, 2) / d)       # one rate per sine/cosine pair
    pe = np.zeros((n_pos, d))
    pe[:, 0::2], pe[:, 1::2] = np.sin(pos * rate), np.cos(pos * rate)
    return pe

pe = sinusoidal(50, 128)                              # 50 positions x 128 dimensions, as on slide 108
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
im = ax[0].imshow(pe, cmap="RdBu", aspect="auto"); ax[0].set_xlabel("dimension"); ax[0].set_ylabel("position")
ax[0].set_title("sinusoidal position encodings"); plt.colorbar(im, ax=ax[0])
for dim in [0, 10, 40, 100]:
    ax[1].plot(pe[:, dim], label=f"dim {dim}")
ax[1].set_xlabel("position"); ax[1].legend(); ax[1].set_title("fast and slow dimensions")
plt.tight_layout(); plt.show()

## 3 · The 8-dimensional table (slide 109)
With d = 8 every number is readable. The encoding is **added** to the word embedding (not concatenated), so the same 8 numbers now carry
both *what* the word is and *where* it is. The word vector below is made up, as on the slide.

In [ ]:
pe8 = sinusoidal(8, 8)
print("pos | " + "  ".join(f"{'sin' if j % 2 == 0 else 'cos'}{j // 2}" for j in range(8)))
for p, row in enumerate(pe8):
    print(f" {p}  | " + "  ".join(f"{v:4.1f}" for v in row))

books = np.array([0.3, -0.5, 0.8, 0.1, -0.2, 0.6, 0.4, -0.7])     # an illustrative embedding of "books"
print("\n'books' at position 1:", books + pe8[1])
print("'books' at position 6:", books + pe8[6])

Compare the columns: the first pair (rate 1 radian per position) changes a lot between rows, while the last pair (rate 0.001) hardly
moves over eight positions — just like the high bit of the counter.

## 4 · Shifting is a rotation, and similarity depends on distance
For each pair, moving from *pos* to *pos + k* rotates the point (sin, cos) by the angle k·ω<sub>i</sub>. That rotation does not depend
on *pos*, so PE(pos + k) is a **fixed linear function** of PE(pos) — the property the original paper hoped would help with relative
positions. As a consequence, the dot product PE(p)·PE(q) depends only on p − q.

In [ ]:
d, k = 128, 5
rates = 10000.0 ** (-np.arange(0, d, 2) / d)
M = np.zeros((d, d))                                   # block-diagonal: one 2x2 rotation per pair
for i, w in enumerate(rates):
    c, s = math.cos(k * w), math.sin(k * w)
    M[2*i:2*i+2, 2*i:2*i+2] = [[c, s], [-s, c]]       # acts on (sin, cos)
pe = sinusoidal(200, d)
print("PE(pos + 5) == M @ PE(pos) for every pos:", np.allclose(pe[5:] , pe[:-5] @ M.T))

sim = pe @ pe.T
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].imshow(sim[:100, :100], cmap="viridis"); ax[0].set_title("PE(p) · PE(q): constant along diagonals")
ax[0].set_xlabel("q"); ax[0].set_ylabel("p")
ax[1].plot(sim[50, :100]); ax[1].axvline(50, ls="--", c="gray"); ax[1].set_title("similarity of position 50 to all others")
ax[1].set_xlabel("position")
plt.tight_layout(); plt.show()

Nearby positions get similar vectors, and the similarity falls off with distance (with wiggles). But the model never sees PE alone:
attention compares *projected* (word + position) vectors, where content and position are mixed. RoPE puts position directly into
the comparison instead.

## 5 · Rotary position embedding (slide 110)
RoPE rotates each (query or key) feature pair by an angle proportional to its position, instead of adding a vector to the input.
Slide 110 uses one pair with a rate of 15° per position: a query with content angle 10° sits at position 2, a key with content angle 40°
at position 5.

In [ ]:
def rot(v, deg):
    t = math.radians(deg)
    return np.array([[math.cos(t), -math.sin(t)], [math.sin(t), math.cos(t)]]) @ v
unit = lambda deg: np.array([math.cos(math.radians(deg)), math.sin(math.radians(deg))])
angle = lambda v: math.degrees(math.atan2(v[1], v[0]))
RATE = 15

q, k = unit(10), unit(40)
for shift in [0, 2]:
    qp, kp = rot(q, RATE * (2 + shift)), rot(k, RATE * (5 + shift))
    print(f"q at {2 + shift}, k at {5 + shift}:  q -> {angle(qp):5.1f} deg,  k -> {angle(kp):5.1f} deg,"
          f"  angle between = {angle(kp) - angle(qp):.1f} deg,  q.k = {qp @ kp:.4f}")

Shifting both positions by 2 rotates both vectors by 30°, so the angle between them (75° = 30° of content + 3 × 15° of distance) and
their dot product stay the same. Now the full version: many pairs with the sinusoidal rates, applied to Q and K of dimension 64.

In [ ]:
def rope(x, pos, base=10000.0):
    # x: [..., d] with d even; rotate pair (2i, 2i+1) by pos * base^(-2i/d)
    d = x.shape[-1]
    theta = pos * base ** (-torch.arange(0, d, 2) / d)
    cos, sin = torch.cos(theta), torch.sin(theta)
    x1, x2 = x[..., 0::2], x[..., 1::2]
    out = torch.empty_like(x)
    out[..., 0::2], out[..., 1::2] = x1 * cos - x2 * sin, x1 * sin + x2 * cos
    return out

torch.manual_seed(0)
q, k = torch.randn(64), torch.randn(64)
for m, n in [(3, 1), (103, 101), (1003, 1001)]:
    print(f"q at {m:4d}, k at {n:4d} (distance {m - n}):  q.k = {rope(q, m) @ rope(k, n):.4f}")
print("norm preserved:", torch.allclose(rope(q, 777).norm(), q.norm()))

The score depends only on the distance m − n, not on where the pair sits in the sequence. A last experiment: if q and k have the
*same* content, how does their score change with distance? The RoFormer paper argues the score tends to **decay** with relative
distance.

In [ ]:
torch.manual_seed(0)
dists = np.arange(0, 512)
scores = np.zeros(len(dists))
for _ in range(100):                                    # average over random content vectors
    v = torch.randn(128)
    scores += np.array([(rope(v, int(t)) @ v).item() for t in dists]) / 100
plt.figure(figsize=(8, 3)); plt.plot(dists, scores)
plt.xlabel("relative distance"); plt.ylabel("q.k (same content)"); plt.title("RoPE: score vs. distance"); plt.show()

**Beyond RoPE.** Slide 111 shows that some 2025 hybrid models (Kimi Linear) use *no* explicit position encoding in their full-attention
layers and rely on recurrent linear-attention layers, which read the sequence in order, to carry position information.

## Exercises
1. Replace `base=10000` by `base=100` in `rope` and redo the decay plot. Why do models that extend their context window often
   *increase* the base?
2. Using `sinusoidal(n, d)`, find the smallest d for which all 1,000 positions get distinct vectors after rounding to one decimal.
3. Verify numerically that adding sinusoidal encodings to Q and K (instead of rotating them) does **not** make q·k depend only on
   m − n. Which extra terms appear when you expand (x + p<sub>m</sub>)ᵀW<sub>q</sub>ᵀW<sub>k</sub>(y + p<sub>n</sub>)?

**Back to the [video lecture]({SITE}#p8-s106)**